# Eurostat SBS: valore aggiunto e dimensione d'impresa

            Questo foglio usa il dataset Eurostat Structural Business Statistics scaricato dalla pipeline. L'obiettivo e passare dalla fonte grezza a viste analitiche su valore aggiunto, imprese, occupazione e produttivita apparente.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from valore_aggiunto_imprese.analysis import (
    METRIC_LABELS,
    add_source_note,
    latest_common_year_with_values,
    latest_year_with_values,
    SIZE_ORDER_BUSINESS,
    SIZE_ORDER_OFFICIAL,
    add_share,
    enrich_business_demography,
    enrich_sbs,
    read_project_csv,
)

pd.options.display.max_columns = 80
pd.options.display.float_format = "{:,.2f}".format

## Base dati

            La tabella raw conserva i codici originali Eurostat. Le funzioni di analisi aggiungono etichette leggibili per paesi, settori, classi e metriche.

In [ ]:
sbs_raw = read_project_csv("data/raw/eurostat_sbs_raw.csv")
sbs = enrich_sbs(sbs_raw)

pd.DataFrame(
    {
        "righe": [len(sbs)],
        "paesi": [sbs["country_code"].nunique()],
        "anni": [f'{int(sbs["year"].min())}-{int(sbs["year"].max())}'],
        "settori": [sbs["sector_code_original"].nunique()],
        "classi": [sbs["size_label"].nunique()],
        "metriche": [sbs["metric_code"].nunique()],
    }
)

## Indicatori disponibili

            Gli indicatori selezionati permettono di leggere scala produttiva, struttura delle imprese, occupazione e produttivita.

In [ ]:
(
    sbs.groupby(["metric_code", "metric_label_readable"], observed=True)
    .agg(righe=("value", "size"), valori_presenti=("value", "count"))
    .reset_index()
    .sort_values("metric_code")
)

## Valore aggiunto per paese e classe

            Il grafico somma i settori inclusi nella configurazione del progetto. La lettura e quindi riferita al perimetro SBS selezionato.

In [ ]:
latest_year = latest_year_with_values(sbs, metric_code="AV_MEUR")
va_latest = (
    sbs.query("metric_code == 'AV_MEUR' and year == @latest_year")
    .dropna(subset=["value"])
    .groupby(["country_label", "size_label"], observed=True, as_index=False)["value"]
    .sum()
)
va_latest["size_label"] = pd.Categorical(
    va_latest["size_label"], SIZE_ORDER_OFFICIAL, ordered=True
)

fig = px.bar(
    va_latest.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="value",
    color="size_label",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"country_label": "Paese", "value": "Milioni di euro", "size_label": "Classe"},
    title=f"Valore aggiunto per classe dimensionale - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Peso relativo delle classi

            Le quote rendono confrontabili paesi con dimensioni economiche diverse e mostrano dove si concentra il valore aggiunto.

In [ ]:
va_share = add_share(va_latest, ["country_label"])
heatmap_data = va_share.pivot_table(
    index="country_label",
    columns="size_label",
    values="share_pct",
    aggfunc="sum",
    observed=True,
).reindex(columns=SIZE_ORDER_OFFICIAL)

fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={"x": "Classe dimensionale", "y": "Paese", "color": "% valore aggiunto"},
    title=f"Quota di valore aggiunto per classe dimensionale - {latest_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Produttivita apparente nella manifattura

            La produttivita apparente e disponibile direttamente in SBS. Qui viene usata sulla manifattura per confrontare il profilo delle classi dimensionali.

In [ ]:
productivity_year = latest_year_with_values(
    sbs, metric_code="LABPRY_TEUR", filters={"sector_code_original": "C"}
)
productivity = (
    sbs.query("metric_code == 'LABPRY_TEUR' and sector_code_original == 'C' and year == @productivity_year")
    .dropna(subset=["value"])
    .copy()
)
productivity["size_label"] = pd.Categorical(
    productivity["size_label"], SIZE_ORDER_OFFICIAL, ordered=True
)

fig = px.bar(
    productivity.sort_values(["country_label", "size_label"]),
    x="country_label",
    y="value",
    color="size_label",
    barmode="group",
    category_orders={"size_label": SIZE_ORDER_OFFICIAL},
    labels={"country_label": "Paese", "value": "Migliaia di euro", "size_label": "Classe"},
    title=f"Produttivita apparente nella manifattura - {productivity_year}",
)
add_source_note(fig, "Eurostat Structural Business Statistics")
fig

## Sintesi tabellare

            La tabella ordina i paesi per quota di valore aggiunto nelle imprese piu grandi.

In [ ]:
(
    va_share.pivot_table(
        index="country_label",
        columns="size_label",
        values="share_pct",
        aggfunc="sum",
        observed=True,
    )
    .reindex(columns=SIZE_ORDER_OFFICIAL)
    .round(1)
    .sort_values("250+", ascending=False)
)